# Hydrostatic pressure and buoyancy

*fluids · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/hydrostatics_buoyancy)

Pressure in a water tank grows linearly with depth, $p = p_0 + \rho_w g h$. Blocks of different densities are dropped in at once (one batched tensor): the light ones bob and float with a fraction $\rho_b/\rho_w$ under water, the heavy ones sink to the floor.

**Vector form**

$$\nabla p = \rho_w\,\mathbf g,\qquad \rho_b V\,\ddot z = \rho_w g\,V_\text{sub} - \rho_b g\,V - \gamma\rho_w V_\text{sub}\,\dot z$$

**Index (tensor) form**

$$p = p_0 + \rho_w\,g_i x_i \qquad \ddot z_b = g\Big(\frac{\rho_w\,d_b}{\rho_b\,s} - 1\Big) - \gamma\,\frac{\rho_w\,d_b}{\rho_b\,s}\,\dot z_b,\quad d_b = \min(\max(-z_b, 0), s)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_i$ | `tank_points(...)` | (nz, nx, 2) | grid of points in the water, component $i \in \{x, z\}$ |
| $g_i$ | `gravity(G, dev)` | (2,) | gravity vector $(0, -g)$ |
| $g_i x_i$ | `einsum('i,...i->...', g, x)` | (nz, nx) | repeated $i$ ⇒ summed: $g\,h$ at depth $h = -z$ |
| $z_b,\ \dot z_b$ | `y[:, 0], y[:, 1]` | (B,) | height of the bottom face and velocity of block $b$ |
| $\rho_b$ | `rho_b` | (B,) | density of block $b$ — the batch index |
| $d_b$ | `submerged_depth(z, s)` | (B,) | submerged depth, between 0 (in the air) and $s$ (fully wet) |
| $\ddot z_b$ | `acceleration(z, v, rho_b, ...)` | (B,) | buoyancy minus weight minus drag, per unit mass |

Each block is a cube of side $s$; the buoyant force $\rho_w g s^2 d_b$ is the weight of the water it displaces (Archimedes), and it is the net result of the pressure $p_0 + \rho_w g h$ pushing harder on the bottom face than on the top. A floating block settles where buoyancy balances weight, $d_b/s = \rho_b/\rho_w$ — ice ($\rho \approx 917$) floats with 92 % under water. The drag is a simple linear damping proportional to the wet volume; real drag is closer to quadratic, and real waves and added mass are ignored. Blocks that sink rest on the floor.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/hydrostatics_buoyancy/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import pressure

DEPTH, P0, RHO_W, G = 1.0, 101_325.0, 1000.0, 9.81  # tank depth (m), air pressure (Pa), water density, gravity


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, s = params["n_blocks"], params["size"]
    width = 0.35 * n
    g = tensors.gravity(G, dev)  # (2,)
    points = tensors.tank_points(width, DEPTH, 40, 60, dev)  # (nz, nx, 2)
    p = pressure(points, g, P0, RHO_W)  # (nz, nx)
    rho_b = tensors.densities(n, params["rho_min"], params["rho_max"], dev)  # (B,)
    y0 = tensors.initial_state(n, 0.15, dev)  # (B, 2)

    times, traj = solve.solve(y0, rho_b, s, DEPTH, params["t_max"], G, RHO_W, params["drag"])

    print(f"pressure at the floor: {p[0, 0]:.0f} Pa = p0 + rho g h = {P0 + RHO_W * G * DEPTH:.0f} Pa")
    for rho, f in zip(rho_b.tolist(), solve.floating_fraction(traj, s).tolist()):
        state = "floats" if rho < RHO_W else "sinks "
        print(f"rho = {rho:6.0f} kg/m³  {state}  submerged fraction {f:.3f}  (Archimedes {min(1, rho / RHO_W):.3f})")
    return plot.animate(times, traj, rho_b, p, width, DEPTH, s, RHO_W, out_dir / "hydrostatics_buoyancy.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/hydrostatics_buoyancy/equation.py
r"""Physics → tensor translation.

Hydrostatic pressure: the pressure grows with depth because the water above has weight,

    \nabla p = \rho_w \mathbf g    ⇒    p(\mathbf x) = p_0 + \rho_w g_i x_i

with g_i = (0, -g) and the water surface at z = 0. The repeated index i is summed (einsum).

Each block b (a cube of side s and density ρ_b) feels its weight, the buoyant force
(Archimedes: the weight of the displaced water) and a damping drag while in the water:

    \rho_b s^3 \ddot z_b = \rho_w g s^2 d_b - \rho_b g s^3 - \gamma \rho_w s^2 d_b\,\dot z_b,   d_b = clamp(-z_b, 0, s)

where z_b is the height of the block's bottom face and d_b its submerged depth. The drag grows with
the wet volume s^2 d_b, like the buoyancy. Dividing by the block mass ρ_b s^3 gives the acceleration
used below; both water forces carry the same factor (ρ_w/ρ_b)(d_b/s).
"""

import torch


def pressure(x: torch.Tensor, g: torch.Tensor, p0: float, rho_w: float) -> torch.Tensor:
    r"""p = p_0 + \rho_w g_i x_i  (i summed).   x: (..., 2) points → (...) pressures in Pa."""
    return p0 + rho_w * torch.einsum("i,...i->...", g, x)


def submerged_depth(z: torch.Tensor, s: float) -> torch.Tensor:
    """How much of each block is under the surface: d_b = clamp(-z_b, 0, s).   z: (B,) → (B,)"""
    return torch.clamp(-z, 0.0, s)


def acceleration(
    z: torch.Tensor, v: torch.Tensor, rho_b: torch.Tensor, s: float, g: float, rho_w: float, gamma: float
) -> torch.Tensor:
    r"""\ddot z_b = g (\rho_w d_b / (\rho_b s) - 1) - \gamma (\rho_w d_b / (\rho_b s)) \dot z_b.   All (B,).

    Buoyancy and drag both act on the wet part of the block only, so both carry the factor ρ_w d_b / (ρ_b s).
    """
    wet = rho_w / rho_b * submerged_depth(z, s) / s  # (B,) displaced water mass / block mass
    return g * (wet - 1.0) - gamma * wet * v


def rhs(y: torch.Tensor, rho_b: torch.Tensor, s: float, g: float, rho_w: float, gamma: float) -> torch.Tensor:
    """dy/dt for the state y of shape (B, 2): y[:, 0] = height z_b of the bottom face, y[:, 1] = velocity."""
    z, v = y[:, 0], y[:, 1]
    return torch.stack([v, acceleration(z, v, rho_b, s, g, rho_w, gamma)], dim=1)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/hydrostatics_buoyancy/tensors.py
"""Build the concrete tensors: gravity, the block densities, their initial state and a grid of tank points."""

import torch


def gravity(g: float, device: torch.device) -> torch.Tensor:
    """g_i with i ∈ {x, z}: shape (2,)."""
    return torch.tensor([0.0, -g], device=device)


def densities(n: int, rho_min: float, rho_max: float, device: torch.device) -> torch.Tensor:
    """n block densities spread evenly between rho_min and rho_max (kg/m³): shape (B,)."""
    return torch.linspace(rho_min, rho_max, n, device=device)


def initial_state(n: int, z0: float, device: torch.device) -> torch.Tensor:
    """y0 of shape (B, 2): every block starts at rest with its bottom face at height z0 above the water."""
    y0 = torch.zeros(n, 2, device=device)
    y0[:, 0] = z0
    return y0


def tank_points(width: float, depth: float, nx: int, nz: int, device: torch.device) -> torch.Tensor:
    """Points x_i filling the water, shape (nz, nx, 2): x from 0 to width, z from -depth up to the surface 0."""
    x = torch.linspace(0.0, width, nx, device=device)
    z = torch.linspace(-depth, 0.0, nz, device=device)
    zz, xx = torch.meshgrid(z, x, indexing="ij")  # (nz, nx) each
    return torch.stack([xx, zz], dim=-1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/hydrostatics_buoyancy/solve.py
"""Integrate all blocks together with RK4; a block that sinks comes to rest on the tank floor."""

import torch

from problems._common.integrators import rk4_step
from problems._common.progress import steps

from .equation import rhs


def solve(
    y0: torch.Tensor,
    rho_b: torch.Tensor,
    s: float,
    depth: float,
    t_max: float,
    g: float = 9.81,
    rho_w: float = 1000.0,
    gamma: float = 4.0,
    dt: float = 0.002,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, B, 2)).

    After every step the floor pushes back: a block below z = -depth is put on the floor with its
    downward velocity removed (the normal force does this in reality).
    """
    n_steps = int(round(t_max / dt))
    y = y0.clone()
    ys = [y]
    for n in steps(n_steps, "RK4 steps"):
        y = rk4_step(lambda y, t: rhs(y, rho_b, s, g, rho_w, gamma), y, n * dt, dt)
        on_floor = y[:, 0] < -depth  # (B,) bool
        y[on_floor, 0] = -depth  # sit on the floor
        y[on_floor, 1] = y[on_floor, 1].clamp(min=0.0)  # no further sinking
        ys.append(y)
    times = torch.arange(n_steps + 1, device=y0.device) * dt
    return times, torch.stack(ys)


def floating_fraction(traj: torch.Tensor, s: float) -> torch.Tensor:
    """Fraction of each block under water at the end of the run: (B,). Archimedes predicts ρ_b / ρ_w."""
    return torch.clamp(-traj[-1, :, 0], 0.0, s) / s


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/hydrostatics_buoyancy/plot.py
"""Left: the tank coloured by pressure with the blocks bobbing. Right: submerged fraction of each block vs time."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Rectangle

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    times: torch.Tensor,
    traj: torch.Tensor,
    rho_b: torch.Tensor,
    p: torch.Tensor,
    width: float,
    depth: float,
    s: float,
    rho_w: float,
    path: Path,
) -> Path:
    t = times.cpu().numpy()  # (T,)
    z = traj[:, :, 0].cpu().numpy()  # (T, B) height of each block's bottom face
    rho = rho_b.cpu().numpy()  # (B,)
    fraction = (-z).clip(0, s) / s  # (T, B)
    n = len(rho)
    gap = width / n
    colors = [ACCENT[b % len(ACCENT)] for b in range(n)]

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(8, 4), gridspec_kw={"width_ratios": [1.5, 1]})
    gauge = (p - p[-1, 0]).cpu().numpy() / 1000  # (nz, nx) gauge pressure p - p0 in kPa
    image = ax.imshow(
        gauge, origin="lower", extent=(0, width, -depth, 0), cmap=plt.get_cmap("Blues", 8), vmin=0,
        vmax=gauge.max() * 1.15, aspect="auto",
    )  # fmt: skip
    fig.colorbar(image, ax=ax, label="p − p₀ (kPa)", shrink=0.8)
    ax.axhline(0, color="#1e40af", lw=1.5)
    ax.set_xlim(0, width)
    ax.set_ylim(-depth, 0.45)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("z (m)")
    ax.grid(False)
    blocks = [
        ax.add_patch(Rectangle(((b + 0.5) * gap - s / 2, z[0, b]), s, s, fc=colors[b], ec="#374151", lw=1))
        for b in range(n)
    ]
    labels = [ax.text((b + 0.5) * gap, 0.0, f"{rho[b]:.0f}", ha="center", fontsize=7) for b in range(n)]
    clock = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top", fontsize=9)

    ax2.set_xlim(0, t[-1])
    ax2.set_ylim(0, 1.1)
    ax2.set_xlabel("t (s)")
    ax2.set_ylabel("submerged fraction")
    ax2.set_title("dashed: Archimedes ρ_b/ρ_w", fontsize=9)
    for b in range(n):
        ax2.axhline(min(1.0, rho[b] / rho_w), color=colors[b], ls="--", lw=0.8)
    curves = [ax2.plot([], [], color=colors[b], lw=1.6)[0] for b in range(n)]

    def update(step: int):
        for b in range(n):
            blocks[b].set_y(z[step, b])
            labels[b].set_y(z[step, b] + s + 0.03)
            curves[b].set_data(t[: step + 1], fraction[: step + 1, b])
        clock.set_text(f"t = {t[step]:.2f} s   (labels: ρ in kg/m³)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path, fps=16)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_blocks': 6,
    'rho_min': 200.0,
    'rho_max': 1400.0,
    'size': 0.2,
    'drag': 4.0,
    't_max': 5.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.hydrostatics_buoyancy")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.hydrostatics_buoyancy.main")
problem_dir = Path("problems/fluids/hydrostatics_buoyancy")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
